# Replication of vanilla RNN with numpy.

### Model is created for prediction of the next element of the sequence based on previous element

At time step $t$, the vanilla RNN updates are:

$$
h_t = \tanh(x_t \cdot W_{xh} + W_{hh} \cdot h_{t-1} + b_h)
$$

$$
y_t = W_{hy} \cdot h_t + b_y
$$


In [2]:
import numpy as np
import sympy as sp

In [ ]:
# Creating the class with forward and backward propagations
class RNN():
    def __init__(self):
        # Initializing params
        self.W_xh, self.W_hh, self.b_h, self.W_hy, self.b_y = 1, 1, 1, 1, 1
        # Making the default initial hidden state
        self.h_init = 1 
    

    # Make it with sympy
    def backward(self, sequence, lr=0.01):
        # Making the scipy symbols as parameters
        W_xh_var, W_hh_var, b_h_var, W_hy_var, b_y_var = sp.symbols('W_xh, W_hh, b_h, W_hy, b_y')
        
        # Initial hidden state as prior
        h_t = self.h_init
        
        # Accuracy counter
        acc_counter = []
        
        # Finding the gradients of each parameter through the timesteps, collecting them in list to sum up
        W_xh_grad_list = []
        W_hh_grad_list = []
        b_h_grad_list = []
        W_hy_grad_list = []
        b_y_grad_list = []
        
        # Create dictionary to map variables with constants
        subs_dict = {
            W_xh_var: self.W_xh,
            W_hh_var: self.W_hh,
            b_h_var: self.b_h,
            W_hy_var: self.W_hy,
            b_y_var: self.b_y
        }
        for i in range(len(sequence)-1):
            h_t = sp.tanh((sequence[i]*W_xh_var+W_hh_var*h_t+b_h_var))
            y_t = W_hy_var*h_t + b_y_var
            y_t_value = y_t.subs(subs_dict).evalf()
            print(f"Predicted: {y_t_value}, Predicted_rounded: {int(round(y_t_value))}, Real: {sequence[i+1]}, {int(round(y_t_value)) == sequence[i+1]}")
            acc_counter.append(int(round(y_t_value)) == sequence[i+1])
            loss_t = (1 / (len(sequence) - 1)) * (y_t - sequence[i+1])**2 # MSE loss
            
            # Finding the gradiens (derivatives, w.r.t. each parameter)
            W_xh_grad_t = sp.diff(loss_t,W_xh_var)
            W_xh_grad_t = W_xh_grad_t.subs(subs_dict).evalf()
            W_xh_grad_list.append(W_xh_grad_t)
            
            W_hh_grad_t = sp.diff(loss_t,W_hh_var)
            W_hh_grad_t = W_hh_grad_t.subs(subs_dict).evalf()
            W_hh_grad_list.append(W_hh_grad_t)
            
            b_h_grad_t = sp.diff(loss_t,b_h_var)
            b_h_grad_t = b_h_grad_t.subs(subs_dict).evalf()
            b_h_grad_list.append(b_h_grad_t)
            
            W_hy_grad_t = sp.diff(loss_t,W_hy_var)
            W_hy_grad_t = W_hy_grad_t.subs(subs_dict).evalf()
            W_hy_grad_list.append(W_hy_grad_t)
            
            b_y_grad_t = sp.diff(loss_t,b_y_var)
            b_y_grad_t = b_y_grad_t.subs(subs_dict).evalf()
            b_y_grad_list.append(b_y_grad_t)
            
        
        # Summing the gradients by timesteps into whole gradients
        W_xh_grad = np.sum(W_xh_grad_list)
        W_hh_grad= np.sum(W_hh_grad_list)
        b_h_grad = np.sum(b_h_grad_list)
        W_hy_grad = np.sum(W_hy_grad_list)
        b_y_grad = np.sum(b_y_grad_list)
        
        # Updating weights and biases
        self.W_xh, self.W_hh, self.b_h, self.W_hy, self.b_y = self.W_xh-lr*W_xh_grad, self.W_hh-lr*W_hh_grad, self.b_h-lr*b_h_grad, self.W_hy-lr*W_hy_grad, self.b_y-lr*b_y_grad
        return sum(acc_counter)

random_sequence = np.array([1,2,4,8,16])
model = RNN()
for i in range(5):
    accuracy = model.backward(random_sequence)
    print(f"Accuracy: {accuracy}")

Predicted: 1.99505475368673, Predicted_rounded: 2, Real: 2, True
Predicted: 1.99932263552791, Predicted_rounded: 2, Real: 4, False
Predicted: 1.99998769499223, Predicted_rounded: 2, Real: 8, False
Predicted: 1.99999999587759, Predicted_rounded: 2, Real: 16, False
Accuracy: 1
Predicted: 2.21456040844687, Predicted_rounded: 2, Real: 2, True
Predicted: 2.21929732906589, Predicted_rounded: 2, Real: 4, False
Predicted: 2.22003542685470, Predicted_rounded: 2, Real: 8, False
Predicted: 2.22004907696999, Predicted_rounded: 2, Real: 16, False
Accuracy: 1
Predicted: 2.42529788495029, Predicted_rounded: 2, Real: 2, True
Predicted: 2.43048535811567, Predicted_rounded: 2, Real: 4, False
Predicted: 2.43129362642787, Predicted_rounded: 2, Real: 8, False
Predicted: 2.43130857314174, Predicted_rounded: 2, Real: 16, False
Accuracy: 1
Predicted: 2.62761720548275, Predicted_rounded: 3, Real: 2, False
Predicted: 2.63323765738894, Predicted_rounded: 3, Real: 4, False
Predicted: 2.63411338169931, Predicted_r

In [14]:
import numpy as np

class RNN:
    def __init__(self, input_size=1, hidden_size=2, output_size=1):
        # Initialize weights and biases with small random values
        self.W_xh = np.random.randn(input_size, hidden_size) * 0.01  # Input to hidden
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.01  # Hidden to hidden
        self.b_h = np.zeros((1, hidden_size))  # Hidden bias
        self.W_hy = np.random.randn(hidden_size, output_size) * 0.01  # Hidden to output
        self.b_y = np.zeros((1, output_size))  # Output bias
        self.hidden_size = hidden_size
        self.h_init = np.zeros((1, hidden_size))  # Initial hidden state

    def forward(self, sequence):
        seq_len = len(sequence)
        h_states = [self.h_init]  # Store hidden states
        outputs = []  # Store predictions
        for t in range(seq_len - 1):
            x_t = np.array([[sequence[t]]])  # Input at time t (1x1)
            h_t = np.tanh(x_t @ self.W_xh + h_states[-1] @ self.W_hh + self.b_h)  # Hidden state update
            y_t = h_t @ self.W_hy + self.b_y  # Output prediction
            h_states.append(h_t)
            outputs.append(y_t)
        return h_states, outputs

    def backward(self, sequence, lr=0.01):
        seq_len = len(sequence) - 1
        h_states, outputs = self.forward(sequence)
        acc_counter = []

        # Gradients for parameters
        dW_xh = np.zeros_like(self.W_xh)
        dW_hh = np.zeros_like(self.W_hh)
        db_h = np.zeros_like(self.b_h)
        dW_hy = np.zeros_like(self.W_hy)
        db_y = np.zeros_like(self.b_y)
        dh_next = np.zeros_like(self.h_init)  # Gradient of next hidden state

        # Backpropagation through time (from last to first timestep)
        for t in reversed(range(seq_len)):
            y_t = outputs[t]
            target = np.array([[sequence[t + 1]]])  # Target at time t+1
            dy = y_t - target  # Gradient of loss w.r.t. output (MSE derivative)
            acc_counter.append(int(round(y_t[0, 0])) == sequence[t + 1])
            print(f"Predicted: {y_t[0, 0]:.2f}, Predicted_rounded: {int(round(y_t[0, 0]))}, Real: {sequence[t+1]}, {int(round(y_t[0, 0])) == sequence[t+1]}")

            # Gradients for output layer
            dW_hy += h_states[t + 1].T @ dy
            db_y += dy

            # Gradient of hidden state at time t
            dh = (dy @ self.W_hy.T) + dh_next
            dh_raw = (1 - h_states[t + 1] ** 2) * dh  # Derivative of tanh

            # Gradients for hidden layer
            x_t = np.array([[sequence[t]]])
            dW_xh += x_t.T @ dh_raw
            dW_hh += h_states[t].T @ dh_raw
            db_h += dh_raw

            # Gradient for next hidden state (for next iteration)
            dh_next = dh_raw @ self.W_hh.T

        # Update weights and biases
        self.W_xh -= lr * dW_xh
        self.W_hh -= lr * dW_hh
        self.b_h -= lr * db_h
        self.W_hy -= lr * dW_hy
        self.b_y -= lr * db_y
        return sum(acc_counter)

# Test the model
random_sequence = np.array([1, 2, 4, 8, 16])
model = RNN()
for epoch in range(10000):
    accuracy = model.backward(random_sequence)
    print(f"Epoch {epoch+1}, Accuracy: {accuracy}")

Predicted: -0.00, Predicted_rounded: 0, Real: 16, False
Predicted: -0.00, Predicted_rounded: 0, Real: 8, False
Predicted: -0.00, Predicted_rounded: 0, Real: 4, False
Predicted: -0.00, Predicted_rounded: 0, Real: 2, False
Epoch 1, Accuracy: 0
Predicted: 0.30, Predicted_rounded: 0, Real: 16, False
Predicted: 0.30, Predicted_rounded: 0, Real: 8, False
Predicted: 0.30, Predicted_rounded: 0, Real: 4, False
Predicted: 0.30, Predicted_rounded: 0, Real: 2, False
Epoch 2, Accuracy: 0
Predicted: 0.61, Predicted_rounded: 1, Real: 16, False
Predicted: 0.60, Predicted_rounded: 1, Real: 8, False
Predicted: 0.59, Predicted_rounded: 1, Real: 4, False
Predicted: 0.59, Predicted_rounded: 1, Real: 2, False
Epoch 3, Accuracy: 0
Predicted: 1.00, Predicted_rounded: 1, Real: 16, False
Predicted: 0.94, Predicted_rounded: 1, Real: 8, False
Predicted: 0.91, Predicted_rounded: 1, Real: 4, False
Predicted: 0.89, Predicted_rounded: 1, Real: 2, False
Epoch 4, Accuracy: 0
Predicted: 1.56, Predicted_rounded: 2, Real: